In [1]:
import chromadb
chroma_client = chromadb.Client()

In [2]:
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

sentence_transformer_ef = SentenceTransformerEmbeddingFunction(
    model_name="multi-qa-MiniLM-L6-cos-v1",
    device="cuda",
    normalize_embeddings=True
)

print("SENTENCE TRANSFORMER")

/run/media/tahas44/Yeni Birim/Technarts/Intern/NLP/InformationRetrieval/.venv/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12240.76it/s]


SENTENCE TRANSFORMER


In [3]:
collection = chroma_client.get_or_create_collection(
    name="IR_multi_qa",
    embedding_function=sentence_transformer_ef
)

In [4]:
import ir_datasets
dataset = ir_datasets.load("wikir/en1k/training")
print("Data set yüklendi")

Data set yüklendi


In [5]:
doc_texts = []
doc_ids = []

for doc in dataset.docs_iter():
    doc_texts.append(doc.text)
    doc_ids.append(doc.doc_id)

### Vector Database Storage

In [6]:
from tqdm import tqdm

BATCH_SIZE = 5000
total_doc = len(doc_texts)

for i in tqdm(range(0, total_doc, BATCH_SIZE)):
    batch_texts = doc_texts[i: i + BATCH_SIZE]
    batch_ids = doc_ids[i: i + BATCH_SIZE]

    collection.add(
        documents=batch_texts,
        ids=batch_ids
    )

print("Vektorization completed!")

100%|██████████| 74/74 [1:22:52<00:00, 67.20s/it]

Vektorization completed!


In [7]:
queries = []

for query in dataset.queries_iter():
    queries.append(query.text)

In [8]:
results_10 = collection.query(
    query_texts=queries,
    n_results=10
)

print("En iyi 10 doküman!")

En iyi 10 doküman!


In [9]:
results_5 = collection.query(
    query_texts=queries,
    n_results=5
)

print("En iyi 5 doküman!")

En iyi 5 doküman!


In [10]:
from collections import defaultdict

qrels_dict = defaultdict(list)

for qrel in dataset.qrels_iter():
    qrels_dict[qrel.query_id].append(qrel.doc_id)

qrels_dict = dict(qrels_dict)

In [11]:
query_ids = [query.query_id for query in dataset.queries_iter()]

In [12]:
class Scoredoc:
    def __init__(self, doc_id, score):
        self.doc_id = doc_id
        self.score = score

score_doc_dict = defaultdict(list)

for scoreddoc in dataset.scoreddocs_iter():
    doc_id = scoreddoc.doc_id
    score = scoreddoc.score

    scoreddoc_object = Scoredoc(doc_id, score)

    score_doc_dict[scoreddoc.query_id].append(scoreddoc_object)

In [13]:
from helper import pipeline
df_parquet = pipeline(results_5, results_10, query_ids, "SenTransformer multi_qa", qrels_dict, score_doc_dict)

     Query_ID   recall_5  precision_5      AP_5    NDCG_5  recall_10  \
0      123839  50.000000         60.0  0.500000  1.000000  66.666667   
1      188629  16.666667         20.0  0.166667  1.000000  16.666667   
2       13898  50.000000         60.0  0.250000  0.000000  50.000000   
3      316959  22.222222         40.0  0.222222  0.979258  22.222222   
4      515031   7.142857         20.0  0.071429  0.707808   7.142857   
...       ...        ...          ...       ...       ...        ...   
1439   896124  12.500000         20.0  0.125000  0.955111  12.500000   
1440    12319   0.000000          0.0  0.000000  0.678174   4.545455   
1441     4421   0.000000          0.0  0.000000  0.866627   6.666667   
1442   296526   0.000000          0.0  0.000000  0.920865   0.000000   
1443   341793  14.285714         20.0  0.142857  0.980100  14.285714   

      precision_10     AP_10   NDCG_10  f_score_5  f_score_10  
0             40.0  0.583333  0.964746  54.545455   50.000000  
1      

In [14]:
df_parquet

,Method,recall_5_mean,recall_5_std,recall_5_max,recall_5_min,recall_10_mean,recall_10_std,recall_10_max,recall_10_min,precision_5_mean,...,MAP_5,MAP_10,NDCG_5_mean,NDCG_5_std,NDCG_5_max,NDCG_5_min,NDCG_10_mean,NDCG_10_std,NDCG_10_max,NDCG_10_min
0,SenTransformer multi_qa,11.927286,11.864656,71.428571,0.0,15.364111,15.377768,100.0,0.0,26.855956,...,0.093419,0.111127,0.715174,0.354878,1.0,0.0,0.714065,0.300219,1.0,0.0


In [15]:
df_parquet.to_parquet("SenTransformerMulti_qa.parquet")